[Home](../../README.md)

### Feature Engineering

This Jupyter Notepad is a selection of data engineering processes you can apply to your data before model training to maximise the performance of your machine learning model. For this demonstration we will engineer new or improved features for the diabetes data you previously wrangled.

#### Feature Engineering Process
- Deriving new variables from existing ones
    - Encoding categorical features
    - Calculating new features from existing features
- Combining features/feature interactions
- Identifying the most relevant features for the model
- Transforming Features
  - [Dividing Data into categories](https://web.ma.utexas.edu/users/mks/statmistakes/dividingcontinuousintocategories.html)
  - Mathematical transformations (for example logarithmic transformations). Logarithmic transformations are a powerful tool in the world of statistical analysis. They are often used to transform data that exhibit skewness or other irregularities, making it easier to analyze, visualize, and interpret the results.
- Creating Domain-Specific Features that incorporating knowledge from the specific domain to create features that capture important characteristics of the data.

#### Load the required dependencies

In [1]:
# Import frameworks
import pandas as pd
from datetime import date

####  Store the data as a local variable

The data frame is a Pandas object that structures your tabular data into an appropriate format. It loads the complete data in memory so it is now ready for preprocessing.

In [2]:
results = pd.read_csv("2.2.1.wrangled_results.csv")
results_all_events = pd.read_csv("2.2.1.unwrangled_results.csv")
persons = pd.read_csv("2.2.1.wrangled_persons.csv")
countries = pd.read_csv("2.2.1.wrangled_countries.csv")
competitions = pd.read_csv("2.2.1.wrangled_competitions.csv")
current_date = date.today().year

#### Calculating Competitions entered

In [3]:
competitions_entered = results_all_events.groupby("person_id")["competition_id"].nunique().reset_index()
competitions_entered.columns = ["person_id", "competitions_entered"]
print(competitions_entered.head())

    person_id  competitions_entered
0  1982BORS01                     1
1  1982BRIN01                     1
2  1982CHIL01                     1
3  1982FRID01                     4
4  1982GALR01                     1


#### Calculating Years Active

In the context of experience and comparing averages of each participant, a cubers years active in the competitions has boundless influence on the target. Thus, we will convert their first competition entered into a year, and minus that from the current year by importing datetime.

In [4]:
results["competition_id"] = results["competition_id"].astype(str)
competitions["id"] = competitions["id"].astype(str)

year_results = results.merge(competitions[["id", "year"]], left_on="competition_id", right_on="id", how="left")
years_active = (year_results.groupby("person_id")["year"].agg(["min", "max"]).reset_index())

years_active["years_active"] = current_date - years_active["min"]
years_active = years_active[["person_id", "years_active"]]
print(years_active.head())

    person_id  years_active
0  1982FRID01            23
1  1982PETR01            22
2  1982RAZO01            22
3  1982VALD01            14
4  2003AKIM01            21


#### Person Country

In [5]:
person_country = year_results.merge(competitions[["id", "country_id"]], left_on="competition_id", right_on="id", how="left")
person_country = person_country.groupby("person_id")["country_id"].agg(lambda avg: avg.mode()[0]).reset_index()
person_country.columns = ["person_id", "country"]
print(person_country.head())

    person_id      country
0  1982FRID01          USA
1  1982PETR01          USA
2  1982RAZO01  Netherlands
3  1982VALD01         Peru
4  2003AKIM01        Japan


#### Person Continent

In [6]:
person_continent = person_country.merge(countries[["id", "continent_id"]], left_on="country", right_on="id", how="left")
person_continent = person_continent[["person_id", "continent_id"]]
person_continent.columns = ["person_id", "continent_id"]
print(person_continent.head())

    person_id    continent_id
0  1982FRID01  _North America
1  1982PETR01  _North America
2  1982RAZO01         _Europe
3  1982VALD01  _South America
4  2003AKIM01           _Asia


## Combining features/feature interactions

While individual features can be powerful predictors, their interactions often carry even more information. Feature interaction engineering is the process of creating new features that represent the interaction between two or more features.

In this case, dividing the total competitions entered within their respected cubing careers (every competition they have entered since their first competition to their last) over their years active in the community (total years since their first competition), derives for us their average competitions per year.

#### Calculating Competitions Per Year


In [7]:
competitions_per_year = competitions_entered.merge(years_active, on="person_id", how="left")
competitions_per_year["competitions_per_year"] = round(competitions_per_year["competitions_entered"] / competitions_per_year["years_active"].replace(0,1), 2)
competitions_per_year = competitions_per_year.dropna(subset=["years_active", "competitions_per_year"])
print(competitions_per_year.head())

     person_id  competitions_entered  years_active  competitions_per_year
3   1982FRID01                     4          23.0                   0.17
8   1982PETR01                    32          22.0                   1.45
9   1982RAZO01                    50          22.0                   2.27
18  1982VALD01                     2          14.0                   0.14
19  2003AKIM01                    19          21.0                   0.90


## Domain specific features

Domain knowledge is about understanding the domain or subject area of the data. In This case the domain is speedcubing and more specifically, in this case, speedcubing algorithms, and the similarities between them. 

Calculating the number of events entered counts more than just the perticipants 3x3x3 entries, as they cold have participated in events such as 2x2x2, 4x4x4, 5x5x5, pyraminx, skewb, and many more. However it is known that many solving algorithms are similar if not exactly the same, no matter the cube. Hence, having experience in other solving categories, before or after having a 3x3x3 attempt, significantly increases the perticipants experience and would therefore decrease their solve time.  

#### Calculating Events Completed

In [8]:
events_completed = results_all_events.groupby("person_id")["event_id"].count().reset_index()
events_completed = events_completed[events_completed["person_id"].isin(results["person_id"])]
events_completed.columns = ["person_id", "events_completed"]
print(events_completed.head())

     person_id  events_completed
3   1982FRID01                 9
8   1982PETR01                91
9   1982RAZO01                95
18  1982VALD01                 2
19  2003AKIM01                58


#### Calculating Events Per Competition  

In [9]:
events_per_competition = competitions_entered.merge(events_completed, on="person_id", how="left")
events_per_competition["events_per_competition"] = round(events_per_competition["events_completed"] / events_per_competition["competitions_entered"].replace(0,1), 2)
events_per_competition = events_per_competition.dropna(subset=["events_completed", "events_per_competition"])
print(events_per_competition.head())

     person_id  competitions_entered  events_completed  events_per_competition
3   1982FRID01                     4               9.0                    2.25
8   1982PETR01                    32              91.0                    2.84
9   1982RAZO01                    50              95.0                    1.90
18  1982VALD01                     2               2.0                    1.00
19  2003AKIM01                    19              58.0                    3.05


#### MERGE


In [10]:
data_frame = results.copy()

data_frame = data_frame.merge(competitions_entered, on="person_id", how="left")
data_frame = data_frame.merge(years_active, on="person_id", how="left")
data_frame = data_frame.merge(events_completed, on="person_id", how="left")
data_frame = data_frame.merge(competitions_per_year[["person_id", "competitions_per_year"]], on="person_id", how="left")
data_frame = data_frame.merge(person_country, on="person_id", how="left")
data_frame = data_frame.merge(person_continent, on="person_id", how="left")
data_frame = data_frame.merge(events_per_competition[["person_id", "events_per_competition"]], on="person_id", how="left")
data_frame = data_frame.drop(columns=["pos", "best", "competition_id", "event_id", "person_name", "person_country_id"])

print(data_frame.head())

   average   person_id  competitions_entered  years_active  events_completed  \
0     2128  2007AMAN01                     2            19                 8   
1     2140  2004ROUA01                     6            22                14   
2     2637  2005SIMO01                    28            21               175   
3     2637  2007MALL01                     6            19                33   
4     2906  2007DESM01                    10            19                29   

   competitions_per_year country continent_id  events_per_competition  
0                   0.11  France      _Europe                    4.00  
1                   0.27  France      _Europe                    2.33  
2                   1.33  France      _Europe                    6.25  
3                   0.32  France      _Europe                    5.50  
4                   0.53  France      _Europe                    2.90  


#### Save the wrangled and engineered data to CSV

In [11]:
data_frame.to_csv('../2.3.Model_Training/2.3.1.model_ready_data.csv', index=False)